In [9]:
import Implementation_OFLP as IO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import math
from pyscipopt import Model, quicksum
scip = Model()

In [ ]:
def points_to_cons(points : list, p_inside : list) -> tuple[list, list]:
    A = []
    b = []
    n = len(p)
    # We want constriants of the form Ax <= b, where x = [x,y]
    for i in range(p):
        x1,y1,x2,y2 = points[i][0], points[i][1], points[(i + 1) % n][0], points[(i + 1) % n][1]
        new_A = [y2 - y1 , x1 - x2]
        new_b = x1 * y2 - x2 * y1
        if new_A @ p_inside > new_b:
            A.append(- new_A)
            b.append(- new_b)
        else:
            A.append(new_A)
            b.append(new_b)
    return A, b

def feas_region_constraint(x_lb, x_ub, y_lb, y_ub, circles, method : int, model, var) -> None:
    '''
    This function creates the constraint for the feasible region.
    :param method: This is either 0 or 1 and it denotes the methods, 0 is LP, and 1 is GS.
    '''
    if method == 0:
        solutions = IO.generate_points(x_lb, x_ub, y_lb, y_ub)
        A,b = IO.feas_region_box_LP(x_lb, x_ub, y_lb, y_ub, circles, solutions)
    sorted_points, deleted_points = IO.feas_region_box_GS(x_lb, x_ub, y_lb, y_ub, circles)
    if len(deleted_points) == 0:
        n = len(sorted_points)
        p1, p2 = sorted_points[0], sorted_points[math.ceil(n/2)]
        x1, x2 = p1[0], p1[1]
        y1, y2 = p2[0], p2[1]
        deleted_points = [[x1 + ((x2 - x1) / 2), y1 + ((y2 - y1) / 2)]]
    A, b = points_to_cons(sorted_points, deleted_points[0])
    for i in range(len(b)):
        model.addCons(A[i] * var <= b[i], f'Convex hull constraint {i} for {var}')
    return None

def distance(p1, p2) -> float:
    return (p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2


def create_objective(model, var_x, var_y, circles):
    model.addVar(L, ub=0, lb=None)
    for c in circles:
        for i, (x, y) in enumerate(zip(var_x, var_y)):
            model.addCons(distance(c,v) <= L, f"MaxDistance_{i}")
    scip.setObjective(L,'maximize')
    return None
    

def create_LP(x_lb : float, x_ub : float, y_lb : float, y_ub : float, circles : list, facilities : int, mode : int) -> None:

    OFP = Model()
    ObnFacilities_x = [OFP.addVar(vtype='C', lb=x_lb, ub=x_ub, name=f"x_{i}") for i in range(facilities)]
    ObnFacilities_y = [OFP.addVar(vtype='C', lb=y_lb, ub=y_ub, name=f"y_{i}") for i in range(facilities)]
    for i in range(facilities):
        var = [ObnFacilities_x[i], ObnFacilities_y[i]]
        feas_region_constraint(x_lb, x_ub, y_lb, y_ub, circles, mode, OFP, var)
    create_objective(model, ObnFacilities_x,ObnFacilities_y, circles)
    

In [30]:
model = Model()
n = 10
x = [
    model.addVar(vtype='C', lb=0, name=f'facility{i}')
    for i in range(n)]

cons_1 = [model.addCons(x[i] + x[(i+1)%10] <= 3, name=f'Cons_{i}') for i in range(n)
        ]
cons_2 = [model.addCons(x[i] - x[(i + 3) %n] == 0) for i in range(3,n)]
    
model.setObjective(np.sum(x), sense='maximize')
model.optimize()

feasible solution found by trivial heuristic after 0.0 seconds, objective value 0.000000e+00
presolving:
(round 1, fast)       7 del vars, 7 del conss, 0 add conss, 10 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
(round 2, fast)       7 del vars, 8 del conss, 0 add conss, 11 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
(round 3, exhaustive) 7 del vars, 14 del conss, 0 add conss, 11 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
   (0.0s) symmetry computation started: requiring (bin +, int +, cont +), (fixed: bin -, int -, cont -)
   (0.0s) symmetry computation finished: 1 generators found (max: 1500, log10 of symmetry group size: 0.30) (symcode time: 0.00)
dynamic symmetry handling statistics:
   orbitopal reduction:       no components
   orbital reduction:         no components
   lexicographic reduction:   no permutations
handled 1 out of 1 symmetry components
(round 4, exhausti

# Running the main program

In [ ]:
def create_LP(sorted_points : list)

4
5
